In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split 
import matplotlib.pyplot as plt

# Lab 8 - Multi-layer Perceptron Forward Pass

## Part I
For this exercise you will implement a simple 2-layer perceptron (forward pass)

For the first part you'll write a function that computes the forward pass of a 2-layer perecptron that predicts the prices of houses, using the usual Boston housing dataset.

In [2]:
housing_names = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"]
boston = pd.read_table("housing.txt", names=housing_names, sep='\s+')

In [3]:
X = boston.values[:,:-1]
X = np.column_stack((np.ones(X.shape[0]), X))
y = boston.values[:,-1]

As usual, consider the MEDV as your target variable. 
* Split the data into training, validation and testing (70,15,15)% (you will need this for the next lab as we will build from this lab)

In [4]:
# 70% train | 15% val | 15% test
X_train, X_aux, y_train, y_aux = train_test_split(X, y, train_size=0.70, random_state=42)
X_val,   X_test,  y_val,  y_test  = train_test_split(X_aux, y_aux, test_size=0.50, random_state=42)

Now you will write the function that computes the forward pass. 
* I provide here a structure that you can follow for your function, but again, feel free to modify it as you see fit.
* Use the sigmoid function as the activation of the hidden layer.
* Don't forget about the biases!
* *It is up to you to think what should be the activation for the output layer.*

In [5]:
def sigmoid_activation(z):
    return 1/(1+np.exp(-z))

In [6]:
def two_layer_perceptron(X, hidden_activation, output_activation,
                         dim_input, dim_hidden, dim_output):
    """
    Forward pass of a 2-layer fully connected perceptron.

    Parameters
    ----------
    X                 : ndarray of shape (N, dim_input)
                        Input data (first column must be the bias column of 1s).
    hidden_activation : callable
                        Activation function applied to the hidden layer.
    output_activation : callable
                        Activation function applied to the output layer
                        (use identity lambda z: z for regression).
    dim_input         : int – number of input features (including bias)
    dim_hidden        : int – number of hidden neurons
    dim_output        : int – number of output neurons

    Returns
    -------
    y_pred : ndarray of shape (N,) if dim_output == 1, else (N, dim_output)
    """
    # Hidden layer  (X already carries the bias column)
    W1 = np.random.randn(dim_input, dim_hidden)
    a1 = hidden_activation(X @ W1)               # (N, dim_hidden)
    a1_bias = np.column_stack((np.ones(X.shape[0]), a1))  # add bias → (N, dim_hidden+1)

    # Output layer
    W2 = np.random.randn(dim_hidden + 1, dim_output)
    y_pred = output_activation(a1_bias @ W2)      # (N, dim_output)

    return y_pred.ravel() if dim_output == 1 else y_pred


In [7]:
identity = lambda z: z   # linear activation → appropriate for regression

np.random.seed(0)
y_pred = two_layer_perceptron(X_train, sigmoid_activation, identity, 14, 30, 1)
y_pred

C:\Users\sodre\AppData\Local\Temp\ipykernel_3384\386652744.py:2: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-z))


array([-1.96869646, -2.73911513, -1.96869648, -3.61042212, -1.32455252,
       -1.32455252, -1.96869646, -1.32455252, -1.96870098, -2.8194611 ,
       -3.61042212, -2.82779296, -2.55091889, -2.81490407, -2.24080703,
       -1.32455252, -2.41232638, -2.40969527, -1.33227824, -2.19357306,
       -2.82779292, -1.33232039, -1.3245526 , -1.32457125, -1.33050952,
       -3.61042152, -2.74145863, -1.96869646, -2.19355844, -1.32455252,
       -1.32455252, -1.32455252, -2.55092106, -1.79824629, -1.79773067,
       -2.19430291, -1.32455252, -3.61042212, -1.32455252, -1.32455252,
       -3.61040328, -1.32455252, -1.32455252, -2.55092106, -3.61042212,
       -2.19443971, -1.32455252, -1.32455252, -2.55097613, -1.33560453,
       -2.82759508, -1.96869651, -2.55445907, -3.61042212, -1.32455252,
       -3.54995119, -2.74145863, -2.82760073, -2.74180079, -2.82779297,
       -1.02515923, -3.61042212, -2.74145342, -2.55114338, -3.61042212,
       -2.74145863, -1.32455252, -1.32455252, -3.61042212, -2.15

Calculate the RMSE of the forward pass. 

In [8]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


In [9]:
rmse(y_train, y_pred)

np.float64(27.050562978294444)

## Part II 

For this exercise you will write a function that calculates the forward pass of a 2-layer perceptron that predicts the exact digit from a hand-written image, using the MNIST dataset. 

In [10]:
from sklearn.datasets import load_digits

In [11]:
digits = load_digits()

In [12]:
X = digits.data
X = np.column_stack((np.ones(X.shape[0]), X))
y = digits.target

In [13]:
X.shape

(1797, 65)

Again, you will split the data into training, validation and testing.

In [14]:
# 70% train | 15% val | 15% test
X_train, X_aux, y_train, y_aux = train_test_split(X, y, train_size=0.70, random_state=42)
X_val,   X_test,  y_val,  y_test  = train_test_split(X_aux, y_aux, test_size=0.50, random_state=42)

Write a function that calculates the forward pass for this multi-class classification problem.
* You will use the sigmoid activation function for the hidden layer.
* For the output layer you will have to write the softmax activation function (you can check the slides)
* __Note:__ you can easily re-use the function that you coded for Part I if you do a simple modification and also include an input argument for the activation of the output layer.

In [15]:
def softmax_activation(z):
    # Subtract row-wise max for numerical stability (avoids exp overflow)
    z_stable = z - z.max(axis=1, keepdims=True)
    exp_z    = np.exp(z_stable)
    return exp_z / exp_z.sum(axis=1, keepdims=True)


In [19]:
# Reuse the same two_layer_perceptron – just swap the output activation
np.random.seed(0)
y_pred_ = two_layer_perceptron(X_train, sigmoid_activation, softmax_activation, 65, 14, 10)
y_pred_


array([[0.00271355, 0.01200222, 0.11544388, ..., 0.01921686, 0.12870065,
        0.04339653],
       [0.00403839, 0.02875441, 0.1896291 , ..., 0.2892065 , 0.02314989,
        0.00332655],
       [0.07444654, 0.01325339, 0.30563234, ..., 0.05937128, 0.06281886,
        0.01149645],
       ...,
       [0.03032251, 0.12382948, 0.30938557, ..., 0.04422421, 0.0299199 ,
        0.0504948 ],
       [0.00379106, 0.017453  , 0.0953964 , ..., 0.37225691, 0.05278172,
        0.00176036],
       [0.00611552, 0.01378056, 0.07632533, ..., 0.10520606, 0.0065751 ,
        0.00389844]], shape=(1257, 10))

In [17]:
# Sanity check: each row is a probability distribution over 10 classes
print('output shape  :', y_pred_.shape)
print('row sums (≈1) :', y_pred_[:3].sum(axis=1))


output shape  : (1257, 10)
row sums (≈1) : [1. 1. 1.]


Lastly, calculate the error of this forward pass using the cross-entropy loss.

In [18]:
def cross_entropy(y_true, y_pred, eps=1e-12):
    """
    Categorical cross-entropy loss.

    Parameters
    ----------
    y_true : ndarray of shape (N,) – integer class labels
    y_pred : ndarray of shape (N, K) – softmax probabilities
    eps    : float – small constant to avoid log(0)

    Returns
    -------
    loss : scalar
    """
    N = y_true.shape[0]
    # Clip predictions to avoid numerical issues with log
    y_pred_clipped = np.clip(y_pred, eps, 1 - eps)
    # For each sample, pick the log-probability of the true class
    log_likelihood = np.log(y_pred_clipped[np.arange(N), y_true])
    return -np.mean(log_likelihood)

ce_loss = cross_entropy(y_train, y_pred_)
print(f'Cross-entropy loss (random weights): {ce_loss:.4f}')
print(f'Expected for random classifier:      {np.log(10):.4f}  (≈ log 10)')


Cross-entropy loss (random weights): 3.5202
Expected for random classifier:      2.3026  (≈ log 10)
